In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from app.data_providers import provide_dataframe, DataFrameRequest
from processing.fixes import fix_score_column, fix_score_columns_for_all_games
import pandas as pd
import sys

In [2]:
request = DataFrameRequest(
    apply_preprocessing = True,
    add_computed = True,
    filter_pre_encoding_columns = False,
    encode_for_model = False,
    filter_top_players=True,
    filter_clean= True
)

# pipeline execution
df = provide_dataframe(request)


Repo card metadata block was not found. Setting CardData to empty.
/Users/mykolapoliakov/PycharmProjects/nba_shot_predictions/processing/preprocessing.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  shot_result_conv = df['shotResult'].replace({'Made': 1, 'Missed': 0})


# Train test split

In [3]:
df = df.reset_index(drop=True)

In [4]:
df = df.sort_values("GAME_DATE")

In [6]:
# Separate our DataSet into a train and a test DataSets. The test DataSet represent 20% of the database.
test_df = df.sample(frac=0.2, random_state=42)
train_df = df.drop(test_df.index)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 430327
Test size: 107582


In [7]:
#train_counts = train_df["PLAYER_ID"].value_counts(normalize=True)
#test_counts = test_df["PLAYER_ID"].value_counts(normalize=True)
train_counts = train_df["PLAYER_ID"].value_counts(normalize=False)
test_counts = test_df["PLAYER_ID"].value_counts(normalize=False)

comparison = pd.concat([train_counts, test_counts], axis=1)
comparison.columns = ["train_share", "test_share"]
comparison = comparison.fillna(0)

print(comparison.sort_values("test_share", ascending=False).head(10))

           train_share  test_share
PLAYER_ID                         
2544.0           40556       10263
977.0            33866        8459
1717.0           28556        7246
201142.0         27293        6946
1495.0           27570        6919
201566.0         26186        6549
201935.0         25655        6437
2548.0           23494        5843
708.0            23451        5806
201939.0         21390        5236


In [8]:
print(comparison['test_share']/(comparison['test_share'] + comparison['train_share']))

PLAYER_ID
2544.0       0.201952
977.0        0.199858
1717.0       0.202391
1495.0       0.200615
201142.0     0.202868
201566.0     0.200061
201935.0     0.200580
2548.0       0.199168
708.0        0.198448
201939.0     0.196650
101108.0     0.194565
2225.0       0.199330
2200.0       0.200256
203507.0     0.201353
203076.0     0.199952
1938.0       0.203259
202695.0     0.196302
203999.0     0.200367
1629029.0    0.195936
203110.0     0.201459
dtype: float64


# Save the files

In [10]:
df.to_parquet('../data/processed/processed_20_players.parquet')
train_df.to_parquet('../data/processed/processed_20_players_train.parquet')
test_df.to_parquet('../data/processed/processed_20_players_test.parquet')